In [ ]:
import pandas as pd
from experiment import RollingTestAnalyzer

raw_path = r"rolling_test_raw.csv"
history_df = pd.read_csv(r"data\power_daily_raw.csv")

rolling_df = pd.read_csv(raw_path)
first_target = pd.to_datetime(rolling_df["target_date"]).min()

# 首个 test target_date 之前的真实历史序列
history_actuals = history_df.loc[
    pd.to_datetime(history_df["date"]) < first_target,
    "OT",
].to_numpy(dtype=float)
   
seasonality = 7

a = RollingTestAnalyzer(
    rolling_raw=r"artifacts\TimeLLM_no_feat\rolling_test_raw.csv",
    history_actuals=history_actuals,
    seasonality=seasonality,
)

b = RollingTestAnalyzer(
    rolling_raw=r"rolling_test_raw.csv",
    history_actuals=history_actuals,
    seasonality=seasonality,
)

# 1. overall 对比
overall_a = a.overall_metrics()
overall_b = b.overall_metrics()

# 2. horizon-wise 对比（手动看两张表）
horizon_a = a.loss_matrix("horizon")
horizon_b = b.loss_matrix("horizon")

# 3. window-wise 对比
window_a = a.loss_matrix("window")
window_b = b.loss_matrix("window")

# A 相对 B 的 window 统计 + win_rate
summary_a_vs_b = a.loss_summary("window", baseline=b)








In [2]:
print(f'a:\n{overall_a}')
print(f'b:\n{overall_b}')



a:
MASE       2.073678e+00
RMSE       4.037640e+08
WAPE(%)    1.859316e+01
Bias       2.190922e+08
MAPE(%)    2.539291e+01
Name: overall, dtype: float64
b:
MASE       1.992035e+00
RMSE       3.862510e+08
WAPE(%)    1.786112e+01
Bias       1.813523e+08
MAPE(%)    2.425429e+01
Name: overall, dtype: float64


In [3]:
print('horizon_a:')
horizon_a


horizon_a:


,MAE,RMSE,Bias
horizon,,,
1,2.668915e+08,3.643390e+08,1.587797e+08
2,3.066505e+08,4.212032e+08,2.429066e+08
3,3.005706e+08,4.194800e+08,2.444487e+08
4,2.810014e+08,3.875680e+08,1.921693e+08
5,2.816162e+08,3.927945e+08,2.048126e+08
6,3.054411e+08,4.191517e+08,2.434590e+08
7,3.027293e+08,4.181798e+08,2.470697e+08


In [4]:
print('horizon_b:')
horizon_b

horizon_b:


,MAE,RMSE,Bias
horizon,,,
1,2.349734e+08,3.125554e+08,5.169137e+07
2,3.037465e+08,4.228523e+08,2.465785e+08
3,2.767153e+08,3.797123e+08,1.777743e+08
4,2.622812e+08,3.563938e+08,1.393594e+08
5,2.811176e+08,3.838858e+08,1.795622e+08
6,3.104593e+08,4.296525e+08,2.581888e+08
7,2.950970e+08,4.057913e+08,2.163117e+08


In [5]:
print(f'window_a:')
window_a

window_a:


,MAE,RMSE,MAPE(%)
origin_date,,,
2024-11-10,3.323948e+08,3.413087e+08,20.375153
2024-11-11,3.859706e+08,4.111451e+08,25.024232
2024-11-12,3.607493e+08,3.863513e+08,23.347067
2024-11-13,3.583395e+08,3.770191e+08,23.015125
2024-11-14,3.452897e+08,3.689188e+08,22.246257
...,...,...,...
2025-01-31,7.415269e+08,7.511950e+08,77.430620
2025-02-01,6.612272e+08,6.830014e+08,66.036989
2025-02-02,5.847749e+08,6.137470e+08,55.489030


In [6]:
print('window_b')
window_b

window_b


,MAE,RMSE,MAPE(%)
origin_date,,,
2024-11-10,2.877445e+08,3.042209e+08,17.642041
2024-11-11,3.253326e+08,3.670137e+08,21.256042
2024-11-12,2.993287e+08,3.416061e+08,19.582997
2024-11-13,2.972574e+08,3.299754e+08,19.213683
2024-11-14,2.896758e+08,3.177265e+08,18.653334
...,...,...,...
2025-01-31,6.935404e+08,7.024420e+08,72.093883
2025-02-01,6.153182e+08,6.317142e+08,60.863353
2025-02-02,5.437241e+08,5.639256e+08,50.884612


In [7]:
summary_a_vs_b

,mean,median,std,worst10%,max,win_rate
MAE,2.921286e+08,1.667656e+08,2.629547e+08,8.650121e+08,8.962177e+08,0.425287
RMSE,3.082126e+08,2.022536e+08,2.608263e+08,8.679568e+08,8.982969e+08,0.413793
MAPE(%),2.539291e+01,1.022897e+01,3.114586e+01,9.970205e+01,1.055423e+02,0.402299
